# Demand Forecasting with Price Sensitivity - Part 1

This notebook builds a demand forecasting model that incorporates price sensitivity to enable revenue optimization.

Part 1: Data Loading, Preprocessing, and Feature Engineering

In [ ]:
pip install numpy pandas matplotlib seaborn calplot mplcyberpunk catboost scikit-learn holidays joblib

In [1]:
import numpy as np
import pandas as pd
import os
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import matplotlib.pyplot as plt
import seaborn as sns
from calplot import calplot as clp
import mplcyberpunk
plt.style.use("cyberpunk")

from catboost import CatBoostRegressor

from sklearn.metrics import root_mean_squared_error as RMSE
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import gc
import holidays

pd.set_option('display.float_format', lambda x: '%.4f' % x)

In [2]:
# Set up the project directory
# Use the notebook's directory as the base path
import notebook
import os

# Get the directory where this notebook is located
try:
    # Try to get notebook directory
    notebook_dir = os.path.dirname(notebook.notebook_dir)
    os.chdir(notebook_dir)
except:
    # Fallback: use current directory
    pass

print("Current folder:", os.getcwd())
print("Directory contents:", os.listdir())

Current folder: /workdir
Directory contents: ['mlc_notebook_readme.md', '.ipynb_checkpoints', 'sber_h1.xlsx', 'eurusd_h1.xlsx', 'gold_h1.xlsx', 'catboost_retail_model.cbm', 'ML_TrendModelGPT_clean.ipynb', 'silver_h1.xlsx', '.virtual_documents', 'catboost_clf_1.cbm', 'mag', 'project_another', 'stores.csv', 'markdowns.csv', 'zoomcamp.ipynb', 'online.csv', 'sample_submission.csv', 'test.csv', 'price_history.csv', 'sales.csv', 'brent_h1.xlsx', 'discounts_history.csv', 'submission_baseline.csv', 'catboost_info', 'submission_catboost_best_clip.csv', 'top4-retail-demand-forecast-mlzc-compet-24.ipynb', 'catboost_clf_old.cbm', 'zoomcamp_modified_version.ipynb', 'submission.csv', 'translated_catalog.csv', 'catalog.csv', 'actual_matrix.csv', 'old_zoomcamp_modified_version_old.ipynb', 'revenue_optimization_implementation.ipynb', 'demand_forecasting_with_price_sensitivity.ipynb', 'price_optimization_analysis.ipynb', 'revenue_optimization_overview.ipynb', 'pricing_strategy.csv', 'demand_forecast_mod

In [3]:
# Load data files
files = {
    "sales": "../data/sales.csv",
    "online": "../data/online.csv",
    "markdowns": "../data/markdowns.csv",
    "price_history": "../data/price_history.csv",
    "discounts_history": "../data/discounts_history.csv",
    "actual_matrix": "../data/actual_matrix.csv",
    "catalog": "../data/catalog.csv",
    "stores": "../data/stores.csv",
    "test": "../data/test.csv",
    "sample_submission": "../data/sample_submission.csv",
}

def read_csv(name, **kwargs):
    fp = files[name]
    df = pd.read_csv(fp, **kwargs)
    return df

df_sales = read_csv("sales", index_col=0)
df_online = read_csv("online", index_col=0)
df_markdowns = read_csv("markdowns", index_col=0)
df_price_history = read_csv("price_history", index_col=0)
df_discounts_history = read_csv("discounts_history", index_col=0)
df_actual_matrix = read_csv("actual_matrix", index_col=0)
df_catalog = read_csv("catalog", index_col=0)
df_stores = read_csv("stores", index_col=0)

df_test = read_csv("test", sep=";", index_col="row_id")
df_test = df_test.reset_index()  # ADDED: row_id теперь обычная колонка и больше не потеряется
df_sample_submission = read_csv("sample_submission", index_col=0)

## Enhanced Data Preparation with Price Features

In [4]:
# Optimize dtypes function
def optimizing_dtypes(df, nameCSV):
    bytesInOneMB = 1048576
    print(f"{nameCSV}: from {round(df.memory_usage().sum()/bytesInOneMB, 2)}MB", end="")

    col_float64 = df.select_dtypes(include=["float64"]).columns.values
    if len(col_float64) > 0:
        df[col_float64] = df[col_float64].astype("float32", copy=False)

    col_int64 = df.select_dtypes(include=["int64"]).columns.values
    if len(col_int64) > 0:
        for col in col_int64:
            uniq = df[col].dropna().unique()
            if len(uniq) <= 2 and set(uniq).issubset({0, 1}):
                df[col] = df[col].astype("int8", copy=False)
            else:
                df[col] = df[col].astype("int32", copy=False)

    print(f" reduced to {round(df.memory_usage().sum()/bytesInOneMB,2)}MB")
    return df

# Optimize dataframes
df_sales = optimizing_dtypes(df_sales, "sales.csv")
df_online = optimizing_dtypes(df_online, "online.csv")
df_markdowns = optimizing_dtypes(df_markdowns, "markdowns.csv")
df_price_history = optimizing_dtypes(df_price_history, "price_history.csv")
df_discounts_history = optimizing_dtypes(df_discounts_history, "discounts_history.csv")
df_actual_matrix = optimizing_dtypes(df_actual_matrix, "actual_matrix.csv")
df_catalog = optimizing_dtypes(df_catalog, "catalog.csv")
df_stores = optimizing_dtypes(df_stores, "stores.csv")
df_test = optimizing_dtypes(df_test, "test.csv")
df_sample_submission = optimizing_dtypes(df_sample_submission, "sample_submission.csv")

sales.csv: from 396.95MB reduced to 283.53MB
online.csv: from 60.0MB reduced to 42.85MB
markdowns.csv: from 0.48MB reduced to 0.34MB
price_history.csv: from 31.98MB reduced to 23.99MB
discounts_history.csv: from 257.27MB reduced to 185.81MB
actual_matrix.csv: from 1.07MB reduced to 0.94MB
catalog.csv: from 15.09MB reduced to 12.58MB
stores.csv: from 0.0MB reduced to 0.0MB
test.csv: from 26.97MB reduced to 20.23MB
sample_submission.csv: from 13.48MB reduced to 7.58MB


In [5]:
# Clean sales data
df_sales["date"] = pd.to_datetime(df_sales["date"])
df_sales["price_base"] = df_sales["sum_total"] / df_sales["quantity"]
df_sales = df_sales.sort_values(["date", "item_id", "store_id"])

mask = (df_sales["quantity"] <= 0) | (df_sales["price_base"] <= 0) | (df_sales["sum_total"] <= 0) | ~np.isfinite(df_sales["price_base"])
df_sales.drop(df_sales[mask].index, axis=0, inplace=True)

# Clean test data
df_test["date"] = pd.to_datetime(df_test["date"])

# Clean online data
df_online["date"] = pd.to_datetime(df_online["date"])
df_online["price_base"] = df_online["sum_total"] / df_online["quantity"]
df_online = df_online.sort_values(["date", "item_id", "store_id"])

mask = (df_online["quantity"] <= 0) | (df_online["price_base"] <= 0) | (df_online["sum_total"] <= 0) | ~np.isfinite(df_online["price_base"])
df_online.drop(df_online[mask].index, axis=0, inplace=True)

/tmp/ipykernel_2002/2766753665.py:10: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_test["date"] = pd.to_datetime(df_test["date"])


In [6]:
# Merge sales and online data
df_online = df_online.rename(columns={"price_base":"price_base_online", "sum_total":"sum_total_online"})
df_online["online"] = True

df = pd.merge(df_sales, df_online, on=["date", "item_id", "store_id"], how="outer", suffixes=("_x", "_y"))
df["quantity"] = df[["quantity_x", "quantity_y"]].sum(axis=1)

# Keep columns as in df_sales
df = df[df_sales.columns.to_list()]
df = df.fillna(0)

In [7]:
# Add price history features
df_price_history["date"] = pd.to_datetime(df_price_history["date"])

# Check columns in price history dataframe
print("Price history columns:", df_price_history.columns.tolist()),
print("Main dataframe columns:", df.columns.tolist())

# Merge price history to get historical price context
df = df.merge(df_price_history, on=["date", "item_id", "store_id"], how="left", suffixes=("", "_history"))

# Проверяем колонки и заполняем price_history
print("Merged dataframe columns:", df.columns.tolist())

# Заполняем пропуски в price_history, используя доступные колонки цены
if 'price_history' in df.columns:
    source_col = 'price'
    if source_col not in df.columns:
        source_col = 'price_history'
    
    if source_col != 'price_base':
        df['price_history'] = df[source_col].fillna(df['price_base'])
    # Если source_col == 'price_base', ничего не делаем
else:
    # Создаем price_history из приоритетного источника
    if 'price' in df.columns:
        df['price_history'] = df['price'].fillna(df['price_base'])
    else:
        df['price_history'] = df['price_base'].copy()

Price history columns: ['date', 'item_id', 'price', 'code', 'store_id']
Main dataframe columns: ['date', 'item_id', 'quantity', 'price_base', 'sum_total', 'store_id']
Merged dataframe columns: ['date', 'item_id', 'quantity', 'price_base', 'sum_total', 'store_id', 'price', 'code']


In [8]:
# Add price change features
df["price_change_from_history"] = (df["price_base"] - df["price_history"]) / df["price_history"]
df["price_change_from_history"] = df["price_change_from_history"].fillna(0)

# Add price level categories
df["price_level"] = pd.qcut(df["price_base"], q=5, labels=["very_low", "low", "medium", "high", "very_high"])
df["price_level"] = df["price_level"].cat.add_categories("unknown").fillna("unknown")

In [9]:
# Process discounts history
df_discounts_history["date"] = pd.to_datetime(df_discounts_history["date"])

mask = (
    (df_discounts_history["sale_price_before_promo"] <= 0) |
    (df_discounts_history["sale_price_time_promo"] <= 0) |
    (df_discounts_history["date"] > df_test["date"].max())
)
df_discounts_history.drop(df_discounts_history[mask].index, axis=0, inplace=True)

df_discounts_history["promo_day"] = True
df_discounts_history["discount_percentage"] = (df_discounts_history["sale_price_before_promo"] - df_discounts_history["sale_price_time_promo"]) / df_discounts_history["sale_price_before_promo"] * 100

columns = df.columns.to_list()
df = pd.merge(
    df, df_discounts_history,
    how="left",
    left_on=["date", "item_id", "store_id"],
    right_on=["date", "item_id", "store_id"],
    suffixes=("_x", "_y")
)

df = df[columns + ["promo_day", "sale_price_before_promo", "sale_price_time_promo", "discount_percentage", "promo_type_code", "doc_id", "number_disc_day"]]
df["promo_day"] = df["promo_day"].fillna(False)

fill_cols = ["sale_price_before_promo", "sale_price_time_promo", "discount_percentage", "number_disc_day"]
df[fill_cols] = df[fill_cols].fillna(0)

In [10]:
# Merge stores data
df = df.merge(df_stores, how="left", on=["store_id"])
df_test = df_test.merge(df_stores, how="left", on=["store_id"])

## Feature Engineering for Price Sensitivity

In [11]:
# Add date features
def date_features(dataframe):
    dataframe["day"] = dataframe["date"].dt.day
    dataframe["month"] = dataframe["date"].dt.month
    dataframe["year"] = dataframe["date"].dt.year
    dataframe["dayofweek"] = dataframe["date"].dt.dayofweek
    dataframe["week"] = dataframe["date"].dt.isocalendar().week.astype(int)
    return dataframe

df = date_features(df)
df_test = date_features(df_test)

In [12]:
# Add cyclic features
def transform2cyclic(dataframe):
    dataframe["day_sin"] = np.sin(2 * np.pi * (dataframe["day"] - 1) / 31)
    dataframe["day_cos"] = np.cos(2 * np.pi * (dataframe["day"] - 1) / 31)

    dataframe["dayofweek_sin"] = np.sin(2 * np.pi * dataframe["dayofweek"] / 6)
    dataframe["dayofweek_cos"] = np.cos(2 * np.pi * dataframe["dayofweek"] / 6)

    dataframe["week_sin"] = np.sin(2 * np.pi * (dataframe["week"] - 1) / 52)
    dataframe["week_cos"] = np.cos(2 * np.pi * (dataframe["week"] - 1) / 52)

    dataframe["month_sin"] = np.sin(2 * np.pi * (dataframe["month"] - 1) / 12)
    dataframe["month_cos"] = np.cos(2 * np.pi * (dataframe["month"] - 1) / 12)
    return dataframe

df = transform2cyclic(df)
df_test = transform2cyclic(df_test)

In [13]:
# Add weekend and holiday features
def get_weekends(dataframe):
    dataframe["is_weekend"] = dataframe["dayofweek"].isin([4, 5, 6])
    return dataframe

def get_sundays(dataframe):
    dataframe["is_sunday"] = dataframe["dayofweek"].eq(6)
    return dataframe

def get_holidays(dataframe):
    RU_holidays = holidays.CountryHoliday("RU", years=[2022, 2023, 2024])
    dataframe["holidays"] = False
    dataframe.loc[dataframe["date"].isin(RU_holidays.keys()), "holidays"] = True
    return dataframe

df = get_weekends(df)
df_test = get_weekends(df_test)

df = get_sundays(df)
df_test = get_sundays(df_test)

df = get_holidays(df)
df_test = get_holidays(df_test)

In [14]:
# Add seasonal features
def get_seasons(dataframe):
    dataframe["season"] = 0
    dataframe.loc[(dataframe["month"] == 4) | (dataframe["month"] == 5), "season"] = 1
    dataframe.loc[(dataframe["month"] >= 6) & (dataframe["month"] <= 8), "season"] = 2
    dataframe.loc[(dataframe["month"] == 9) | (dataframe["month"] == 10), "season"] = 3
    dataframe.loc[((dataframe["month"] >= 1) & (dataframe["month"] <= 3)) | (dataframe["month"] == 11) | (dataframe["month"] == 12), "season"] = 4
    return dataframe

df = get_seasons(df)
df_test = get_seasons(df_test)

In [15]:
# Add lag features for price sensitivity
def add_lag_features(df):
    # Sort by date for proper lag calculation
    df = df.sort_values(['item_id', 'store_id', 'date']).reset_index(drop=True)
    
    # Add lagged price features
    df['price_lag_1'] = df.groupby(['item_id', 'store_id'])['price_base'].shift(1)
    df['price_lag_7'] = df.groupby(['item_id', 'store_id'])['price_base'].shift(7)
    df['price_lag_30'] = df.groupby(['item_id', 'store_id'])['price_base'].shift(30)
    
    # Add price change from lags
    df['price_change_1'] = (df['price_base'] - df['price_lag_1']) / df['price_lag_1']
    df['price_change_7'] = (df['price_base'] - df['price_lag_7']) / df['price_lag_7']
    df['price_change_30'] = (df['price_base'] - df['price_lag_30']) / df['price_lag_30']
    
    # Fill NaN values
    price_lag_cols = ['price_lag_1', 'price_lag_7', 'price_lag_30', 'price_change_1', 'price_change_7', 'price_change_30']
    df[price_lag_cols] = df[price_lag_cols].fillna(0)
    
    return df

df = add_lag_features(df)

In [16]:
# Save intermediate data for next part
# IMPORTANT: Save price_base before removing it for modeling
df.to_pickle('df_processed_part1.pkl')
df_test.to_pickle('df_test_processed_part1.pkl')
print("Data saved for next part")

Data saved for next part
